# Household Spending on Alcohol and Tobacco in the Philippines
### Exploratory Data Analysis of the PSA Family Income and Expenditure Survey (FIES) 2023

**Goal:** Understand how Filipino households spend on "vices" (alcohol and tobacco): how common it is, how much is spent, who spends the most relative to their budget, and whether spending can be predicted from household characteristics.

**Data:** `FIES PUF 2023 Volume1.CSV` from the Philippine Statistics Authority (PSA). One row = one household. 163,268 households, 90 columns.

**Important framing:** `ALCOHOL` and `TOBACCO` are **annual household expenditures in pesos**, not individual smoking or drinking behaviour. A household with ₱0 did not *buy* these items during the survey; it does not mean nobody in the household smokes or drinks.

**Survey weights:** each household represents many others. All population-level numbers use the household weight `RFACT` (sums to ~27.5M households).

## 1. Data Loading and Initial Inspection

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

In [ ]:
# Load the dataset
df = pd.read_csv("FIES PUF 2023 Volume1.CSV")
print(df.shape)
df.head()

In [ ]:
# Column types and memory usage
df.info()

In [ ]:
# Summary statistics for the columns this analysis focuses on
key_cols = ["ALCOHOL", "TOBACCO", "TOINC", "TOTEX", "FOOD", "NFOOD", "FSIZE", "PERCAPITA", "RFACT"]
df[key_cols].describe().T

In [ ]:
# Sanity check: the household weights should add up to roughly 27.5 million households
print(f"Weighted number of households: {df['RFACT'].sum():,.0f}")

**Insight:** 
Based on the data loading and inspection, there are 163,268 households that were interviewed or sampled during the data gathering phase. Each of these households represents another X number of households, which is what the column RFACT stands for. This is why the weighted number of households reaches about 27.5 million.

I can say in particular that the use of alcohol or tobacco cannot be separated from the Filipino household. For alcohol, most Filipino households spend their money here, and it is more affordable than tobacco, which will be explained later. For Filipino households in 2023, half of the households are said to be spending 340 or less on alcohol per year. For tobacco, a Filipino household splurges way more money. Even though the median is 0, the top quarter is spending at least 1,820 per year, compared to alcohol, where the top quarter is only spending at least 1,620. With this, fewer households buy tobacco, but the ones that do spend more than alcohol buyers. A question we can formulate here is: can only households above the poverty line afford tobacco products?

But based on this data loading alone, I cannot yet say whether alcohol and tobacco are directly intertwined with the financial state of a family. These findings are only based on the 163,268 households and do not yet represent the weighted number of households, which is about 27.5 million.

## 2. Data Cleaning and Feature Engineering

In [ ]:
# Check for missing values and duplicate households
print("Missing values:", df.isna().sum().sum())
print("Duplicate households:", df.duplicated(["W_REGN", "W_PROV", "SEQ_NO"]).sum())

In [ ]:
# Confirm the accounting identities so we know which totals already contain vice spending
# TOTEX = FOOD + NFOOD, and FOOD = FOOD_HOME + FOOD_OUTSIDE
# ALCOHOL and TOBACCO are NOT inside FOOD_HOME, so they sit inside NFOOD and TOTEX
# TODO: assert these identities hold (allowing for small rounding differences)

In [ ]:
# Map region codes to names
# TODO: verify every code against the PSA data dictionary before relying on it
REGION_NAMES = {
    1: "Ilocos Region", 2: "Cagayan Valley", 3: "Central Luzon", 4: "CALABARZON",
    5: "Bicol Region", 6: "Western Visayas", 7: "Central Visayas", 8: "Eastern Visayas",
    9: "Zamboanga Peninsula", 10: "Northern Mindanao", 11: "Davao Region", 12: "SOCCSKSARGEN",
    13: "NCR", 14: "CAR", 16: "Caraga", 17: "MIMAROPA", 19: "BARMM",
}
URB_NAMES = {1: "Urban", 2: "Rural"}

# TODO: create df["region"] and df["area"] from the mappings above

In [ ]:
# Drop a column that is almost always zero (spelling is from the source file)
# TODO: drop ALCOHOL_PROCDUCTION_SERVICES

In [ ]:
# Create vice spending features
# TODO:
#   VICE          = ALCOHOL + TOBACCO
#   vice_share    = VICE / TOTEX      (also alc_share, tob_share)
#   buys_alcohol, buys_tobacco, buys_any  (spending > 0)

In [ ]:
# Create income and budget features
# TODO:
#   log_percapita = log of PERCAPITA
#   wage_share    = WAGES / TOINC
#   remit_share   = CASH_ABROAD / TOINC
#   pension_share = PENSION / TOINC
#   food_share, health_share, education_share = category / TOTEX

In [ ]:
# Helper: weighted average using the household weight RFACT
def wmean(data, col, weight="RFACT"):
    return np.average(data[col], weights=data[weight])

**Insight:** _TODO_

## 3. Exploratory Data Analysis (EDA)

### 3.1 How common is vice spending?

In [ ]:
# Weighted % of households buying alcohol only, tobacco only, both, or neither
# TODO

**Insight:** _TODO_

### 3.2 How much do buying households spend?

In [ ]:
# Distribution of spending among buyers (log scale) and national totals in billions of pesos
# TODO

**Insight:** _TODO_

### 3.3 Vice spending by income decile

In [ ]:
# Two panels by NPCINC decile: pesos spent vs share of the budget
# TODO

**Insight:** _TODO_

### 3.4 Geography: region and urban vs rural

In [ ]:
# Purchase rate and vice share by region (sorted horizontal bars), plus urban vs rural
# TODO

**Insight:** _TODO_

### 3.5 Household size

In [ ]:
# Vice spending per household and per member against FSIZE
# TODO

**Insight:** _TODO_

### 3.6 Income source: remittances, wages, and business income

In [ ]:
# Compare households with and without OFW remittances, and wage vs entrepreneurial households
# TODO

**Insight:** _TODO_

### 3.7 Crowding out: do heavy vice spenders spend less on education and health?

In [ ]:
# Within each income decile, compare education and health shares of heavy spenders vs non-buyers
# TODO

**Insight:** _TODO_

### 3.8 Alcohol vs tobacco

In [ ]:
# Overlap between alcohol and tobacco buyers, and a scatter of amounts among households buying both
# TODO

**Insight:** _TODO_

## 4. Statistical Checks

With 163k households almost every test will be "significant", so focus on the size of the differences rather than the p-values.

In [ ]:
# Chi-square: buys_any vs region, and buys_any vs urban/rural
# TODO

In [ ]:
# Kruskal-Wallis: vice_share across income deciles; Mann-Whitney U: urban vs rural
# TODO

**Insight:** _TODO_

## 5. Feature Preparation

**Leakage warning:** `TOTEX`, `NFOOD`, and `TOTDIS` already contain alcohol and tobacco spending, so they must not be used as features.

In [ ]:
# Choose features and targets
# TODO: FEATURES = ["log_percapita", "FSIZE", "region", "area", "wage_share", "remit_share", ...]
LEAKAGE_COLS = ["TOTEX", "NFOOD", "TOTDIS", "ALCOHOL", "TOBACCO", "VICE",
                "vice_share", "alc_share", "tob_share"]
# TODO: assert no column in FEATURES is in LEAKAGE_COLS

In [ ]:
# One-hot encode categorical columns
# TODO

In [ ]:
# Split into training and testing sets (stratified), keeping RFACT as sample weights
# TODO

## 6. Machine Learning

Expected spending can be thought of in two parts: **P(household buys)** × **amount spent if it buys**. We model each part separately.

### 6.1 Classification: does the household buy tobacco?

In [ ]:
# Train and evaluate Logistic Regression, Random Forest, and HistGradientBoosting
# Metrics: ROC-AUC, F1, confusion matrix
# TODO

In [ ]:
# Interpret the logistic regression as odds ratios
# TODO

**Insight:** _TODO_

### 6.2 Regression: how much do buying households spend?

In [ ]:
# Among households with VICE > 0, predict log(VICE)
# Train and evaluate Linear Regression, Random Forest, and HistGradientBoosting
# Metrics: RMSE, MAE, R²
# TODO

In [ ]:
# Permutation importance for the best-performing model
# TODO

**Insight:** _TODO_

### 6.3 Clustering: what kinds of spending profiles do households have?

Group households by **how they split their budget**, then check how vice spending differs between the groups. Vice shares are left out of the clustering features on purpose, so the clusters describe household lifestyles and vice spending is compared across them afterwards.

In [ ]:
# Choose clustering features and scale them
# TODO:
#   CLUSTER_FEATURES = budget shares of TOTEX (food_home, food_outside, housing, health, education,
#                      transport, communication, recreation) + log_percapita + FSIZE
#   Standardize with StandardScaler so no single feature dominates the distances

In [ ]:
# Pick the number of clusters
# TODO: elbow plot (inertia) for k = 2..10, and silhouette score on a random sample of ~10,000 households
#       (silhouette on all 163k rows is very slow)

In [ ]:
# Fit K-Means with the chosen k and add the cluster label to df
# TODO: KMeans(n_clusters=k, n_init=10, random_state=42), passing RFACT as sample_weight

In [ ]:
# Profile each cluster
# TODO: weighted % of households, mean feature values, income decile mix, and urban/rural split per cluster
#       then give each cluster a short descriptive name (e.g. "urban wage earners", "rural farm households")

In [ ]:
# Compare vice spending across clusters
# TODO: weighted buys_alcohol, buys_tobacco, and vice_share by cluster (bar charts)

In [ ]:
# Visualize the clusters in 2D
# TODO: PCA to 2 components, scatter a sample of households coloured by cluster

**Insight:** _TODO_

## 7. Summary and Recommendations

### Key findings
- _TODO_

### Limitations
- Expenditure is not the same as consumption; vice spending is commonly under-reported in surveys.
- Cross-sectional data: relationships are associations, not causes.
- Volume 1 has no household-head characteristics (sex, age, education, occupation), which limits the models.

### Next steps
- Merge FIES 2023 Volume 2 on `W_REGN`, `W_PROV`, `SEQ_NO` to add household-head characteristics.
- _TODO_